In [1]:
try:
    import pulp
except ImportError:
    import sys
    !{sys.executable} -m pip install pulp

In [2]:
import numpy as np
from scipy.stats import poisson
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, value

# Parameters
total_rooms = 118
low_fare = 159
high_fare = 225
mean_demand_high_fare = 27.3

# Poisson probabilities
probabilities = [poisson.pmf(d, mean_demand_high_fare) for d in range(total_rooms + 1)]

# Define the LP problem
problem = LpProblem("Maximize_Expected_Revenue", LpMaximize)

# Decision variables
protect = LpVariable("protect", 0, total_rooms, cat="Integer")
sales_high = LpVariable.dicts("sales_high", range(total_rooms + 1), 0, total_rooms, cat="Continuous")

# Objective function
revenue_terms = []

for d in range(total_rooms + 1):
    revenue_terms.append(probabilities[d] * (sales_high[d] * high_fare + (total_rooms - protect) * low_fare))

problem += lpSum(revenue_terms), "Expected_Revenue"

# Constraints
for d in range(total_rooms + 1):
    problem += sales_high[d] <= protect, f"High_Fare_Sales_Limit_{d}"
    problem += sales_high[d] <= d, f"Demand_Limit_{d}"

# Solve the problem
problem.solve()

# Extract results
optimal_protected_high_fare_rooms = int(value(protect))
max_revenue = value(problem.objective)


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/s_/tc52sgkj18d_pnr8h48bmz3m0000gn/T/4d466983954f49bfbacf1cb6a310177a-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/s_/tc52sgkj18d_pnr8h48bmz3m0000gn/T/4d466983954f49bfbacf1cb6a310177a-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 243 COLUMNS
At line 723 RHS
At line 962 BOUNDS
At line 1083 ENDATA
Problem MODEL has 238 rows, 120 columns and 357 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1404.88 - 0.00 seconds
Cgl0004I processed model has 81 rows, 82 columns (11 integer (1 of which binary)) and 162 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of -1404.88
Cbc0038I Relaxing cont

In [3]:

print(f"Optimal number of high fare rooms to protect: {optimal_protected_high_fare_rooms}")
print(f"Maximum expected revenue: ${max_revenue:.2f}")

Optimal number of high fare rooms to protect: 24
Maximum expected revenue: $20166.88
